# TripWorld destination-input catalog

Build **only the destination/city autocomplete catalog**. Past-visit POI search, place-name cleaning, category filtering, recommendation candidates, and model integration are deferred until the teammate's cleaned POI list is available.

The destination IDs come from every distinct `travel_behaviors.r_o`. No destination is removed for low activity or unknown model coverage. Preserve the source's city/country labels and published municipality aliases, including Kyoto → Osaka (Keihanshin).

**Output:** `backend/artifacts/tripworld_destinations.sqlite`. The executed notebook, source profile, manifest, audit, and validation evidence are saved locally. No GPU, reviews, POI metadata, or credentials are needed.

## 1. Repository and pinned dependencies

Use a Python 3.12 CPU runtime and **Run all**. Local execution writes into this checkout. In Colab, set `REPO_REF` to a published branch/commit containing these scripts, or upload the repository into `/content/cmpe-256-final-project`. Unpushed workspace changes are not available to a GitHub clone.

Only three data files are downloaded: `travel_behaviors.parquet`, `region_labels.parquet`, and `metro_mapping_clean.json` (about 145 MiB total). Allow 1 GiB of free disk. The builder reads only `r_o` and the three needed region-label columns.

In [1]:
from pathlib import Path
import importlib.metadata
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/isaackimmi/cmpe-256-final-project.git"
REPO_REF = os.environ.get("TRIPWORLD_REPO_REF", "main")
candidates = [Path.cwd(), Path.cwd().parent, Path("/content/cmpe-256-final-project")]
ROOT = next((p.resolve() for p in candidates if (p / "scripts/build_tripworld_catalog.py").exists()), None)
if ROOT is None:
    ROOT = Path("/content/cmpe-256-final-project")
    if ROOT.exists():
        raise RuntimeError("Upload the catalog scripts into the existing checkout, or choose a fresh checkout directory.")
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=ROOT, check=True)
    if not (ROOT / "scripts/build_tripworld_catalog.py").exists():
        raise RuntimeError("Choose a published branch containing the destination-catalog scripts, or upload the repository.")
requirements = ROOT / "scripts/requirements-catalog.txt"
needed = False
for line in requirements.read_text().splitlines():
    package, version = line.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    needed |= installed != version
if needed:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)
print("Python:", sys.version)
print("Scope: destination input only; no POI cleaning or model assumptions.")

Repository: /workspace/cmpe-256-destination-integration
Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Scope: destination input only; no POI cleaning or model assumptions.


## 2. Profile, build, and validate

The source revision is pinned to `254f142dfef42104783754019f7b7f4cfbf8d9a8`. All three data files must match the upstream SHA-256 manifest. Preserve upstream redistribution documentation and provenance scripts; verify those against their pinned Git blobs because the upstream README checksum is stale.

Left-join labels onto all source destination IDs. Missing or raw-ID labels are flagged and retained. Conflicting label records block release; exact duplicates collapse. Aliases target existing destination IDs, and same-name cities remain separate results with country context.

Names retain their original scripts. There is no name-language or ASCII filter and no Google-category filter. Search preserves Unicode and folds Latin accents; it uses the published names and aliases without inventing translations.

The completed temporary SQLite file is checked in a fresh Python process. Only a successful validation can replace the final artifact.

In [2]:
command = [sys.executable, str(ROOT / "scripts/build_tripworld_catalog.py")]
subprocess.run(command, cwd=ROOT, check=True)
manifest_path = ROOT / "reports/tripworld_catalog_manifest.json"
audit_path = ROOT / "reports/tripworld_catalog_audit.json"
validation_path = ROOT / "reports/tripworld_catalog_validation.json"
manifest = json.loads(manifest_path.read_text())
audit = json.loads(audit_path.read_text())
validation = json.loads(validation_path.read_text())
print(json.dumps({key: manifest[key] for key in ["scope", "source_revision", "database", "database_size_bytes", "counts", "database_sha256", "logical_sha256", "model_coverage"]}, indent=2))
print("Source profile:", json.dumps(audit["profile"], indent=2))
print("Destination ID differences:", validation["checks"]["destination_id_set_exact"])

Checking three pinned destination sources; no POIs, metadata, or reviews are read.


{
  "scope": "destinations_only",
  "passed": true,
  "latency": {
    "queries": 200,
    "median_ms": 0.152,
    "p95_ms": 0.296,
    "max_ms": 0.92,
    "query_cycle": [
      "Kyoto",
      "Kyo",
      "London",
      "Paris",
      "New York",
      "gulluk",
      "zzzzzzzzzzzznomatch"
    ],
    "conditions": "Warm SQLite/OS caches; seven warmup queries; one read-only connection; serial calls; excludes API/network time.",
    "python": "3.12.14",
    "sqlite": "3.53.1",
    "platform": "Linux-6.18.57-x86_64-with-glibc2.41"
  }
}
Acceptance checks: 27
{
  "scope": "destinations_only",
  "database": "backend/artifacts/tripworld_destinations.sqlite",
  "database_size_bytes": 425984,
  "counts": {
    "destinations": 1165,
    "canonical_names": 1165,
    "mapping_names_deduplicated": 901,
    "mapping_entries_outside_destinations": 5201,
    "mapping_entries_without_searchable_label": 256,
    "mapping_aliases_added": 344
  },
  "logical_sha256": "c8f4b5eb2e41a7c9aebad5d8606b1d7cf

## 3. Destination search examples

`Catalog` opens the SQLite file read-only and checks FTS5 in the actual serving runtime. Queries are parameterized and converted into safe token-prefix expressions. Results are distinct by region ID; exact canonical names rank before matching aliases, then prefix matches. Country words can disambiguate same-name cities.

Names and their scripts are preserved, but local-language translations and Japanese/Thai word segmentation are not added. These examples use the dataset's published labels and aliases. Empty queries browse the canonical catalog; punctuation-only or over-200-character queries return no results. The API defaults to 20 items per page, permits up to 100, and supports offset pagination.

In [3]:
from backend.destination_catalog import Catalog
DATABASE = ROOT / "backend/artifacts/tripworld_destinations.sqlite"

def search_destinations(query, limit=10):
    with Catalog(DATABASE) as catalog:
        return catalog.search_destinations(query, limit)

for query in ["Kyoto", "Kyo", "London", "New York", "zzzzzzzzzzzznomatch"]:
    print(query, "→", json.dumps(search_destinations(query), ensure_ascii=False, indent=2))
print("Source trip counts describe travel activity, not POI/model coverage.")

Kyoto → [
  {
    "region_id": "Q35765",
    "city_name": "Osaka",
    "country_name": "Japan",
    "canonical_metro_label": "Osaka (Keihanshin)",
    "label_missing": 0,
    "source_trip_count": 19726,
    "matched_name": "Kyoto",
    "matched_alias": "Kyoto"
  }
]
Kyo → [
  {
    "region_id": "Q35765",
    "city_name": "Osaka",
    "country_name": "Japan",
    "canonical_metro_label": "Osaka (Keihanshin)",
    "label_missing": 0,
    "source_trip_count": 19726,
    "matched_name": "Kyoto",
    "matched_alias": "Kyoto"
  }
]
London → [
  {
    "region_id": "Q84",
    "city_name": "London",
    "country_name": "United Kingdom",
    "canonical_metro_label": "London",
    "label_missing": 0,
    "source_trip_count": 1176,
    "matched_name": "London",
    "matched_alias": null
  },
  {
    "region_id": "Q23311",
    "city_name": "City of London",
    "country_name": "United Kingdom",
    "canonical_metro_label": "City of London",
    "label_missing": 0,
    "source_trip_count": 198,
    

## 4. Acceptance evidence and latency

Validation requires exact equality with the source destination IDs, verifies every published searchable alias, preserves canonical labels, tests Kyoto's metro scope, partial names, accents, same-name cities, country context, safe punctuation, and the least-active source destination. It also requires that the database contains no POI tables or recommendation coverage claims.

`source_trip_count` is the number of source travel records for a destination. Model availability and recommendation counts are **not evaluated** until the cleaned POI/model IDs are available.

In [4]:
print("Checks passed:", len(validation["checks"]))
print("All passed:", validation["passed"])
print(json.dumps(validation["examples"], ensure_ascii=False, indent=2))
print(json.dumps(validation["latency"], indent=2))

Checks passed: 27
All passed: True
{
  "Kyoto": [
    {
      "region_id": "Q35765",
      "city_name": "Osaka",
      "country_name": "Japan",
      "canonical_metro_label": "Osaka (Keihanshin)",
      "label_missing": 0,
      "source_trip_count": 19726,
      "matched_name": "Kyoto",
      "matched_alias": "Kyoto"
    }
  ],
  "Kyo": [
    {
      "region_id": "Q35765",
      "city_name": "Osaka",
      "country_name": "Japan",
      "canonical_metro_label": "Osaka (Keihanshin)",
      "label_missing": 0,
      "source_trip_count": 19726,
      "matched_name": "Kyoto",
      "matched_alias": "Kyoto"
    }
  ],
  "Accent-insensitive query": {
    "query": "gulluk",
    "results": [
      {
        "region_id": "Q1004900",
        "city_name": "Güllük",
        "country_name": "Turkey",
        "canonical_metro_label": "Güllük",
        "label_missing": 0,
        "source_trip_count": 195,
        "matched_name": "Güllük",
        "matched_alias": null
      }
    ]
  },
  "Same-name 

## 5. Reproduce the destination catalog

Run the full build again from the verified source cache and require identical sorted logical tables, cleaning counts, and source-ID coverage. The logical checksum excludes SQLite-generated alias row IDs and report timing metadata.

In [5]:
first_checksum = manifest["logical_sha256"]
first_counts = manifest["counts"]
subprocess.run(command, cwd=ROOT, check=True)
manifest = json.loads(manifest_path.read_text())
audit = json.loads(audit_path.read_text())
validation = json.loads(validation_path.read_text())
assert manifest["logical_sha256"] == first_checksum
assert manifest["counts"] == first_counts
assert audit["same_as_previous_build"]
print("Reproducible logical SHA-256:", first_checksum)
print("Destinations:", manifest["counts"]["destinations"])
print("Final SQLite size (KiB):", round(DATABASE.stat().st_size / 1024, 2))

Checking three pinned destination sources; no POIs, metadata, or reviews are read.


{
  "scope": "destinations_only",
  "passed": true,
  "latency": {
    "queries": 200,
    "median_ms": 0.15,
    "p95_ms": 0.271,
    "max_ms": 0.369,
    "query_cycle": [
      "Kyoto",
      "Kyo",
      "London",
      "Paris",
      "New York",
      "gulluk",
      "zzzzzzzzzzzznomatch"
    ],
    "conditions": "Warm SQLite/OS caches; seven warmup queries; one read-only connection; serial calls; excludes API/network time.",
    "python": "3.12.14",
    "sqlite": "3.53.1",
    "platform": "Linux-6.18.57-x86_64-with-glibc2.41"
  }
}
Acceptance checks: 27
{
  "scope": "destinations_only",
  "database": "backend/artifacts/tripworld_destinations.sqlite",
  "database_size_bytes": 425984,
  "counts": {
    "destinations": 1165,
    "canonical_names": 1165,
    "mapping_names_deduplicated": 901,
    "mapping_entries_outside_destinations": 5201,
    "mapping_entries_without_searchable_label": 256,
    "mapping_aliases_added": 344
  },
  "logical_sha256": "c8f4b5eb2e41a7c9aebad5d8606b1d7cf

## 6. Download or use the backend artifact

Local execution already writes to `backend/artifacts/tripworld_destinations.sqlite`. In Colab, download the ZIP and extract it into your local repository root. The backend can then use `Catalog().search_destinations(...)`; the backend serves this data through `/destinations/search` using `items` and `next_offset`.

The bundle contains the destination database, source profile, manifest, audit, validation output, summary, and upstream redistribution provenance. The small validated SQLite file and upstream provenance are tracked with the backend so a Render code deployment includes them. Raw data, temporary databases, and ZIP downloads remain ignored.

POI search and recommendation candidates are separate from this export. The backend still uses sample POIs and recommendations until the cleaned POI/model integration is ready.

In [6]:
import zipfile
bundle = ROOT / "data/tripworld_destinations_bundle.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    files = [DATABASE, manifest_path, audit_path, validation_path,
             ROOT / "reports/tripworld_catalog_profile.json", ROOT / "reports/tripworld_catalog_summary.md"]
    files += sorted((ROOT / "backend/artifacts/tripworld_provenance").rglob("*"))
    for path in files:
        if path.is_file():
            archive.write(path, path.relative_to(ROOT))
print("Bundle:", bundle)
print("Compressed size (KiB):", round(bundle.stat().st_size / 1024, 2))
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(bundle))

Bundle: /workspace/cmpe-256-destination-integration/data/tripworld_destinations_bundle.zip
Compressed size (KiB): 243.5
